# Universal Audio to Audio Converter
Transcode audio files with maximum flexibility:
- **Single Audio File**
- **Multiple Selected Audio Files** (hold Ctrl/Shift in browser)
- **Full Folder / Directory** (batch conversion)

### Supported Audio Formats (21 Containers / Codecs):
`.mp3`, `.wav`, `.aac`, `.m4a`, `.flac`, `.ogg`, `.opus`, `.wma`, `.aiff`, `.aif`, `.alac`, `.amr`, `.mid`, `.midi`, `.ac3`, `.eac3`, `.ape`, `.dsf`, `.dff`, `.caf`, `.au`

---
### Key Features:
- **Interactive Multi-Mode Selector**: Choose 1 audio file, multiple files, or full directory.
- **Universal Codec Engine**: Lossless PCM/FLAC/ALAC, high-efficiency Opus/AAC/OGG, or standard MP3/WMA transcoding.
- **Audio Quality Profiles**: Audiophile 320k, 256k, 192k standard, 128k compact, or uncompressed PCM.
- **Batch Processing with Progress**: Real-time progress monitoring, summary table, and JSON report generation.

## Step 1: Environment & FFmpeg Verification
Checks for `ffmpeg` availability.

In [1]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

def detect_ffmpeg():
    # 1. System PATH
    exe = shutil.which("ffmpeg")
    if exe:
        return exe
        
    # 2. Local venv
    venv_exe = Path(r"d:\New folder\PY\.venv\Scripts\ffmpeg.exe")
    if venv_exe.is_file():
        os.environ["PATH"] = str(venv_exe.parent) + os.pathsep + os.environ.get("PATH", "")
        return str(venv_exe)

    # 3. imageio_ffmpeg bundled binary
    try:
        import imageio_ffmpeg
        loc = imageio_ffmpeg.get_ffmpeg_exe()
        if loc and os.path.isfile(loc):
            os.environ["PATH"] = str(Path(loc).parent) + os.pathsep + os.environ.get("PATH", "")
            return loc
    except Exception:
        pass

    # 4. Common locations
    common_locations = [
        r"C:\ffmpeg\bin\ffmpeg.exe",
        r"C:\ffmpeg\ffmpeg.exe",
        r"C:\Program Files\ffmpeg\bin\ffmpeg.exe",
        r"C:\ProgramData\chocolatey\bin\ffmpeg.exe",
        os.path.expanduser(r"~\AppData\Local\Microsoft\WinGet\Links\ffmpeg.exe"),
        os.path.expanduser(r"~\AppData\Local\ffmpeg\bin\ffmpeg.exe"),
    ]
    for loc in common_locations:
        if os.path.isfile(loc):
            os.environ["PATH"] = str(Path(loc).parent) + os.pathsep + os.environ.get("PATH", "")
            return loc

    return None

FFMPEG_PATH = detect_ffmpeg()
if FFMPEG_PATH:
    print(f"FFmpeg detected successfully: {FFMPEG_PATH}")
else:
    print("WARNING: FFmpeg was not detected in PATH.")
    print("Run: pip install imageio-ffmpeg")


FFmpeg detected successfully: d:\New folder\PY\.venv\Scripts\ffmpeg.EXE


## Step 2: Define Supported Audio Formats & Transcoder Matrix
List of 21 audio formats and command builders.

In [2]:
AUDIO_FORMATS = [
    ".mp3", ".wav", ".aac", ".m4a", ".flac", ".ogg", ".opus", ".wma",
    ".aiff", ".aif", ".alac", ".amr", ".mid", ".midi", ".ac3", ".eac3",
    ".ape", ".dsf", ".dff", ".caf", ".au"
]

def build_audio_ffmpeg_args(target_ext, bitrate="192k"):
    ext = target_ext.lower()
    args = []
    
    if ext == ".mp3":
        args.extend(["-c:a", "libmp3lame", "-b:a", bitrate])
    elif ext == ".wav":
        args.extend(["-c:a", "pcm_s16le"])
    elif ext in (".aac", ".m4a"):
        args.extend(["-c:a", "aac", "-b:a", bitrate])
    elif ext == ".flac":
        args.extend(["-c:a", "flac"])
    elif ext == ".ogg":
        args.extend(["-c:a", "libvorbis", "-b:a", bitrate])
    elif ext == ".opus":
        opus_br = "128k" if bitrate in ("192k", "128k") else ("160k" if bitrate == "256k" else "192k")
        args.extend(["-c:a", "libopus", "-b:a", opus_br])
    elif ext == ".wma":
        args.extend(["-c:a", "wmav2", "-b:a", "160k"])
    elif ext in (".aiff", ".aif"):
        args.extend(["-c:a", "pcm_s16be"])
    elif ext == ".alac":
        args.extend(["-c:a", "alac"])
    elif ext == ".amr":
        args.extend(["-c:a", "amr_nb", "-ar", "8000", "-ac", "1"])
    elif ext == ".ac3":
        args.extend(["-c:a", "ac3", "-b:a", "384k"])
    elif ext == ".eac3":
        args.extend(["-c:a", "eac3", "-b:a", "448k"])
    elif ext == ".ape":
        args.extend(["-c:a", "ape"])
    elif ext == ".caf":
        args.extend(["-c:a", "pcm_s16le"])
    elif ext == ".au":
        args.extend(["-c:a", "pcm_s16be"])
    elif ext in (".dsf", ".dff"):
        args.extend(["-c:a", "copy"])
    else:
        args.extend(["-b:a", bitrate])
        
    return args

print(f"Configured {len(AUDIO_FORMATS)} audio formats for universal audio-to-audio conversion.")


Configured 21 audio formats for universal audio-to-audio conversion.


## Step 3: Interactive Selection (Single File, Multiple Files, or Full Folder)
Run this cell to select your input:
- Option [1]: Select **multiple files** (hold Ctrl or Shift in the browser window).
- Option [2]: Select a **single file**.
- Option [3]: Select a **full folder**.
- Option [4]: Type or paste path(s) manually.

In [3]:
import tkinter as tk
from tkinter import filedialog

def pick_media_inputs(supported_formats, media_type="audio"):
    selected_files = []
    source_dir = None
    is_folder = False

    try:
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        print("\nChoose input selection mode:")
        print("  [1] Select MULTIPLE files (Hold Ctrl/Shift to choose multiple files)")
        print("  [2] Select a SINGLE file")
        print("  [3] Select a FULL FOLDER (Process all matching files in directory)")
        print("  [4] Type or paste path(s) manually")
        choice = input("Enter choice [1/2/3/4] (default: 1): ").strip()
        
        ext_filter = [(f"{media_type.title()} Files", "*" + ";*".join(supported_formats)), ("All Files", "*.*")]
        
        if choice == "2":
            single_path = filedialog.askopenfilename(title=f"Select Single {media_type.title()} File", filetypes=ext_filter)
            if single_path:
                p = Path(single_path)
                selected_files = [p]
                source_dir = p.parent
        elif choice == "3":
            folder_path = filedialog.askdirectory(title=f"Select Folder Containing {media_type.title()} Files")
            if folder_path:
                source_dir = Path(folder_path)
                is_folder = True
        elif choice == "4":
            pass
        else:
            multi_paths = filedialog.askopenfilenames(title=f"Select One or Multiple {media_type.title()} Files", filetypes=ext_filter)
            if multi_paths:
                selected_files = [Path(p) for p in multi_paths]
                source_dir = selected_files[0].parent
        root.destroy()
    except Exception:
        pass
        
    if not selected_files and not is_folder:
        raw = input(f"Enter {media_type} path(s) or folder (separate multiple files with comma): ").strip().strip('"').strip("'")
        if raw:
            if "," in raw:
                parts = [p.strip().strip('"').strip("'") for p in raw.split(",") if p.strip()]
            elif ";" in raw:
                parts = [p.strip().strip('"').strip("'") for p in raw.split(";") if p.strip()]
            else:
                parts = [raw]

            if len(parts) == 1 and Path(parts[0]).is_dir():
                is_folder = True
                source_dir = Path(parts[0])
            else:
                selected_files = [Path(p) for p in parts]
                source_dir = selected_files[0].parent

    return selected_files, is_folder, source_dir

SELECTED_FILES, IS_FOLDER, SOURCE_DIR = pick_media_inputs(AUDIO_FORMATS, media_type="audio")
if not IS_FOLDER and not SELECTED_FILES:
    raise ValueError("No valid input files or folder selected.")

if IS_FOLDER:
    print(f"\nSelection Mode: Full Folder -> {SOURCE_DIR}")
elif len(SELECTED_FILES) == 1:
    print(f"\nSelection Mode: Single File -> {SELECTED_FILES[0].name}")
else:
    print(f"\nSelection Mode: Multiple Files -> {len(SELECTED_FILES)} files selected in {SOURCE_DIR}")



Choose input selection mode:
  [1] Select MULTIPLE files (Hold Ctrl/Shift to choose multiple files)
  [2] Select a SINGLE file
  [3] Select a FULL FOLDER (Process all matching files in directory)
  [4] Type or paste path(s) manually

Selection Mode: Full Folder -> D:\CSE Fest\Dance-টিকাটুলির মোড়


## Step 4: Select Audio Input Filter & Target Output Format
Filter audio input files and choose target audio output container.

In [4]:
print("\n" + "=" * 55)
print("SELECT AUDIO INPUT FILTER")
print("=" * 55)
print("  [0] ALL Supported Audio Formats (*.*)")
for idx, ext in enumerate(AUDIO_FORMATS, 1):
    print(f"  [{idx:2d}] {ext}")
print("=" * 55)

in_choice = input("Enter input format choice [0 for all, or 1-21, or type extension] (default: 0): ").strip()
if not in_choice or in_choice == "0":
    SELECTED_INPUT_FILTER = "all"
elif in_choice.isdigit() and 1 <= int(in_choice) <= len(AUDIO_FORMATS):
    SELECTED_INPUT_FILTER = AUDIO_FORMATS[int(in_choice) - 1]
elif in_choice.startswith(".") and in_choice.lower() in AUDIO_FORMATS:
    SELECTED_INPUT_FILTER = in_choice.lower()
elif ("." + in_choice.lower()) in AUDIO_FORMATS:
    SELECTED_INPUT_FILTER = "." + in_choice.lower()
else:
    print(f"Unrecognized choice '{in_choice}'. Defaulting to 'all'.")
    SELECTED_INPUT_FILTER = "all"

print(f"\nInput filter selected: {SELECTED_INPUT_FILTER.upper()}")

print("\n" + "=" * 55)
print("SELECT TARGET AUDIO OUTPUT FORMAT")
print("=" * 55)
for idx, ext in enumerate(AUDIO_FORMATS, 1):
    print(f"  [{idx:2d}] {ext}")
print("=" * 55)

out_choice = input("Enter target audio format [1-21, or type extension e.g. .mp3] (default: 1 [.mp3]): ").strip()
if not out_choice or out_choice == "1":
    TARGET_OUTPUT_FORMAT = ".mp3"
elif out_choice.isdigit() and 1 <= int(out_choice) <= len(AUDIO_FORMATS):
    TARGET_OUTPUT_FORMAT = AUDIO_FORMATS[int(out_choice) - 1]
elif out_choice.startswith(".") and out_choice.lower() in AUDIO_FORMATS:
    TARGET_OUTPUT_FORMAT = out_choice.lower()
elif ("." + out_choice.lower()) in AUDIO_FORMATS:
    TARGET_OUTPUT_FORMAT = "." + out_choice.lower()
else:
    print(f"Unrecognized choice '{out_choice}'. Defaulting to '.mp3'.")
    TARGET_OUTPUT_FORMAT = ".mp3"

print(f"Target audio format selected: {TARGET_OUTPUT_FORMAT}")



SELECT AUDIO INPUT FILTER
  [0] ALL Supported Audio Formats (*.*)
  [ 1] .mp3
  [ 2] .wav
  [ 3] .aac
  [ 4] .m4a
  [ 5] .flac
  [ 6] .ogg
  [ 7] .opus
  [ 8] .wma
  [ 9] .aiff
  [10] .aif
  [11] .alac
  [12] .amr
  [13] .mid
  [14] .midi
  [15] .ac3
  [16] .eac3
  [17] .ape
  [18] .dsf
  [19] .dff
  [20] .caf
  [21] .au

Input filter selected: ALL

SELECT TARGET AUDIO OUTPUT FORMAT
  [ 1] .mp3
  [ 2] .wav
  [ 3] .aac
  [ 4] .m4a
  [ 5] .flac
  [ 6] .ogg
  [ 7] .opus
  [ 8] .wma
  [ 9] .aiff
  [10] .aif
  [11] .alac
  [12] .amr
  [13] .mid
  [14] .midi
  [15] .ac3
  [16] .eac3
  [17] .ape
  [18] .dsf
  [19] .dff
  [20] .caf
  [21] .au
Target audio format selected: .aac


## Step 5: Configure Audio Quality & Output Directory
Set target bitrate and output directory.

In [5]:
print("\nSelect Audio Bitrate / Quality:")
print("  [1] 320 kbps (Very High / Audiophile)")
print("  [2] 256 kbps (High Quality)")
print("  [3] 192 kbps (Standard High Quality - Default)")
print("  [4] 128 kbps (Good / Smaller File Size)")
br_choice = input("Enter choice [1/2/3/4] (default: 3): ").strip()

AUDIO_BITRATE = "192k"
if br_choice == "1":
    AUDIO_BITRATE = "320k"
elif br_choice == "2":
    AUDIO_BITRATE = "256k"
elif br_choice == "4":
    AUDIO_BITRATE = "128k"

# Gather matching files
if IS_FOLDER:
    if SELECTED_INPUT_FILTER == "all":
        target_exts = set(AUDIO_FORMATS)
    else:
        target_exts = {SELECTED_INPUT_FILTER}
    
    files_to_convert = [
        f for f in SOURCE_DIR.iterdir()
        if f.is_file() and f.suffix.lower() in target_exts and f.suffix.lower() != TARGET_OUTPUT_FORMAT
    ]
    if not files_to_convert:
        files_to_convert = [
            f for f in SOURCE_DIR.rglob("*")
            if f.is_file() and f.suffix.lower() in target_exts and f.suffix.lower() != TARGET_OUTPUT_FORMAT
        ]
else:
    if SELECTED_INPUT_FILTER != "all":
        files_to_convert = [
            f for f in SELECTED_FILES
            if f.is_file() and f.suffix.lower() == SELECTED_INPUT_FILTER and f.suffix.lower() != TARGET_OUTPUT_FORMAT
        ]
    else:
        files_to_convert = [
            f for f in SELECTED_FILES
            if f.is_file() and f.suffix.lower() in AUDIO_FORMATS and f.suffix.lower() != TARGET_OUTPUT_FORMAT
        ]

if not files_to_convert:
    raise ValueError(f"No audio files matching filter '{SELECTED_INPUT_FILTER}' found for conversion.")

clean_out_name = f"converted_audio_{TARGET_OUTPUT_FORMAT.lstrip('.')}"
OUTPUT_DIR = SOURCE_DIR / clean_out_name
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nReady to convert {len(files_to_convert)} audio file(s).")
print(f"Target Bitrate: {AUDIO_BITRATE}")
print(f"Destination folder: {OUTPUT_DIR}")



Select Audio Bitrate / Quality:
  [1] 320 kbps (Very High / Audiophile)
  [2] 256 kbps (High Quality)
  [3] 192 kbps (Standard High Quality - Default)
  [4] 128 kbps (Good / Smaller File Size)

Ready to convert 1 audio file(s).
Target Bitrate: 128k
Destination folder: D:\CSE Fest\Dance-টিকাটুলির মোড়\converted_audio_aac


## Step 6: Audio Transcoding Engine with Progress Tracking
Converts audio using FFmpeg with real-time progress.

In [6]:
import time
import re
import json

def get_duration(file_path, ffmpeg_bin):
    if not ffmpeg_bin:
        return None
    try:
        proc = subprocess.run(
            [ffmpeg_bin, "-hide_banner", "-i", str(file_path)],
            stderr=subprocess.PIPE,
            stdout=subprocess.DEVNULL,
            text=True,
            encoding="utf-8",
            errors="replace"
        )
        m = re.search(r"Duration:\s*(\d+):(\d+):(\d+(?:\.\d+)?)", proc.stderr)
        if m:
            h, mins, s = float(m.group(1)), float(m.group(2)), float(m.group(3))
            return h * 3600 + mins * 60 + s
    except Exception:
        pass
    return None

def convert_audio_to_audio(src_file, dst_file, target_ext, ffmpeg_bin="ffmpeg", bitrate="192k"):
    cmd = [ffmpeg_bin, "-y", "-i", str(src_file)]
    audio_flags = build_audio_ffmpeg_args(target_ext, bitrate)
    cmd.extend(audio_flags)
    cmd.extend(["-nostats", "-progress", "pipe:1", str(dst_file)])
    
    duration_sec = get_duration(src_file, ffmpeg_bin)
    start_time = time.time()
    last_reported = -1
    log_tail = []

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        universal_newlines=True,
        bufsize=1
    )

    for line in proc.stdout:
        line_str = line.strip()
        if not line_str:
            continue
        log_tail.append(line_str)
        if len(log_tail) > 100:
            log_tail.pop(0)

        if duration_sec and line_str.startswith("out_time_ms="):
            try:
                ms_val = float(line_str.split("=", 1)[1])
                curr_sec = ms_val / 1_000_000.0
                pct = min(100.0, (curr_sec / duration_sec) * 100.0)
                whole = int(pct)
                if whole >= last_reported + 10:
                    last_reported = whole
                    print(f"    Progress: {pct:5.1f}% [{curr_sec:.1f}s / {duration_sec:.1f}s]")
            except Exception:
                pass
        elif line_str == "progress=end":
            print("    Progress: 100.0% [Completed]")

    ret = proc.wait()
    if ret != 0:
        tail_msg = "\n".join(log_tail[-20:])
        raise RuntimeError(f"FFmpeg conversion failed (code {ret}):\n{tail_msg}")

    elapsed = time.time() - start_time
    return {
        "input": str(src_file),
        "output": str(dst_file),
        "elapsed_seconds": round(elapsed, 2),
        "input_size_bytes": src_file.stat().st_size,
        "output_size_bytes": dst_file.stat().st_size if dst_file.exists() else 0,
        "status": "Success"
    }

conversion_records = []
total_count = len(files_to_convert)

print(f"\nStarting audio conversion for {total_count} file(s)...\n" + "=" * 55)
for i, fpath in enumerate(files_to_convert, 1):
    out_filename = f"{fpath.stem}{TARGET_OUTPUT_FORMAT}"
    dest_path = OUTPUT_DIR / out_filename
    print(f"[{i}/{total_count}] Converting: {fpath.name} -> {out_filename}")
    try:
        rec = convert_audio_to_audio(
            src_file=fpath,
            dst_file=dest_path,
            target_ext=TARGET_OUTPUT_FORMAT,
            ffmpeg_bin=FFMPEG_PATH,
            bitrate=AUDIO_BITRATE
        )
        conversion_records.append(rec)
        print(f"  OK: Finished in {rec['elapsed_seconds']}s")
    except Exception as exc:
        print(f"  ERROR on {fpath.name}: {exc}")
        conversion_records.append({
            "input": str(fpath),
            "output": str(dest_path),
            "status": "Failed",
            "error": str(exc)
        })

print("=" * 55 + "\nAll audio conversion tasks completed!")



Starting audio conversion for 1 file(s)...
[1/1] Converting: tikatuli song.mp3 -> tikatuli song.aac
    Progress:  12.4% [31.3s / 252.5s]
    Progress:  25.3% [63.9s / 252.5s]
    Progress:  46.3% [116.9s / 252.5s]
    Progress:  59.0% [149.1s / 252.5s]
    Progress:  71.1% [179.5s / 252.5s]
    Progress:  83.8% [211.5s / 252.5s]
    Progress:  96.2% [243.0s / 252.5s]
    Progress: 100.0% [Completed]
  OK: Finished in 4.3s
All audio conversion tasks completed!


## Step 7: Summary & Audit Report
View before/after size comparisons and export summary report.

In [7]:
total_in_bytes = 0
total_out_bytes = 0
total_time = 0

print("\n" + "=" * 70)
print(f"{'Source File':<28} | {'In Size':<10} | {'Out Size':<10} | {'Ratio':<6} | {'Time'}")
print("-" * 70)

for rec in conversion_records:
    if rec.get("status") == "Success":
        in_mb = rec["input_size_bytes"] / (1024 * 1024)
        out_mb = rec["output_size_bytes"] / (1024 * 1024)
        ratio = (out_mb / in_mb) if in_mb > 0 else 1.0
        fname = Path(rec["input"]).name
        if len(fname) > 27:
            fname = fname[:24] + "..."
        print(f"{fname:<28} | {in_mb:7.2f} MB | {out_mb:7.2f} MB | {ratio:5.2f}x | {rec['elapsed_seconds']}s")
        total_in_bytes += rec["input_size_bytes"]
        total_out_bytes += rec["output_size_bytes"]
        total_time += rec["elapsed_seconds"]
    else:
        fname = Path(rec["input"]).name
        print(f"{fname:<28} | FAILED: {rec.get('error', 'Unknown')[:35]}")

print("=" * 70)
tot_in_mb = total_in_bytes / (1024 * 1024)
tot_out_mb = total_out_bytes / (1024 * 1024)
print(f"Total Input Size:  {tot_in_mb:.2f} MB")
print(f"Total Output Size: {tot_out_mb:.2f} MB")
if tot_in_mb > 0:
    print(f"Overall Ratio:     {(tot_out_mb / tot_in_mb):.2f}x")
print(f"Total Time Taken:  {total_time:.2f} seconds")
print(f"Destination Folder: {OUTPUT_DIR}")

# Save JSON report
report_path = OUTPUT_DIR / "audio_conversion_summary.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump({
        "conversion_type": "audio_to_audio",
        "target_format": TARGET_OUTPUT_FORMAT,
        "input_filter": SELECTED_INPUT_FILTER,
        "bitrate": AUDIO_BITRATE,
        "records": conversion_records
    }, f, indent=2)
print(f"Saved audit report to: {report_path}")



Source File                  | In Size    | Out Size   | Ratio  | Time
----------------------------------------------------------------------
tikatuli song.mp3            |    5.48 MB |    3.99 MB |  0.73x | 4.3s
Total Input Size:  5.48 MB
Total Output Size: 3.99 MB
Overall Ratio:     0.73x
Total Time Taken:  4.30 seconds
Destination Folder: D:\CSE Fest\Dance-টিকাটুলির মোড়\converted_audio_aac
Saved audit report to: D:\CSE Fest\Dance-টিকাটুলির মোড়\converted_audio_aac\audio_conversion_summary.json
